In [1]:
import pandas as pd
import re
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

In [2]:
BASE_URL = "https://books.toscrape.com/"
LIST_PATTERN = "https://books.toscrape.com/catalogue/page-{}.html"
TIMEOUT = 10
MAX_RETRIES = 3
DELAY_FOR_REQUESTS = 0.2
PAGES = 5
CSV_FILE_NAME = "books.csv"

In [3]:
def fetch(session: requests.Session, url: str):
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = session.get(url, timeout=TIMEOUT)
            r.raise_for_status()
            r.encoding = "utf-8"
            return r.text
        except requests.RequestException as e:
            last_err = e
            print(f"  [!] {url} — попытка {attempt}/{MAX_RETRIES} не удалась: {e}")
            if attempt < MAX_RETRIES:
                print(f"      повтор через {TIMEOUT}s...")
                time.sleep(TIMEOUT)
    raise RuntimeError(f"Не удалось получить {url}: {last_err}")

In [4]:
def parse_listing(html: str):
    soup = BeautifulSoup(html, "html.parser")

    items = []
    for pod in soup.select("article.product_pod"):
        a = pod.select_one("h3 > a")
        title = (a["title"].strip() if a and a.has_attr("title")
                 else (a.get_text(strip=True) if a else None))
        href = a["href"].strip() if a else None
        link = urljoin(LIST_PATTERN.format(1), href) if href else None

        price = None
        p_el = pod.select_one("p.price_color")
        if p_el:
            m = re.search(r"\d+(?:\.\d+)?", p_el.get_text())
            if m:
                price = float(m.group())

        RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

        rating = None
        r_el = pod.select_one("p.star-rating")
        if r_el:
            cls = [c for c in r_el.get("class", []) if c != "star-rating"]
            if cls:
                rating = RATING_MAP.get(cls[0])

        availability = None
        av_el = pod.select_one("p.instock.availability")
        if av_el:
            availability = av_el.get_text(" ", strip=True)

        id = re.sub(r'.*_(\d+)/index\.html.*', r'\1', link)
        
        items.append({
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability,
            "link": link,
            "ID": id
        })
    return items

In [5]:
def find_category(session: requests.Session, link: str):
    html = fetch(session, link)
    soup = BeautifulSoup(html, "html.parser")
    
    category = None
    crumbs = soup.select("ul.breadcrumb li")
    if len(crumbs) >= 3:
        a = crumbs[2].select_one("a")
        if a:
            category = a.get_text(strip=True)

    return category

In [6]:
session = requests.Session()

## Получение всех html страниц

In [7]:
all_rows = []

for page in range(1, PAGES + 1):
    url = LIST_PATTERN.format(page)
    html = fetch(session, url)
    batch = parse_listing(html)
    all_rows.extend(batch)
    time.sleep(DELAY_FOR_REQUESTS)

## Получение категории книг

In [8]:
for i, row in enumerate(all_rows, 1):
    if row["link"]:
        cat = find_category(session, row["link"])
        row["category"] = cat
    if i % 20 == 0:
        print(f"{i}/{len(all_rows)}")
    time.sleep(DELAY_FOR_REQUESTS)

20/100
40/100
60/100
80/100
100/100


## Очистка пробелов, приведение числовых полей к числовому типу и удаление дубликатов.

In [9]:
for row in all_rows:
    for k, v in row.items():
        if isinstance(v, str):
            row[k] = re.sub(r"\s+", " ", v).strip()
    if row.get("price") is not None:
        row["price"] = float(row["price"])
    if row.get("rating") is not None:
        row["rating"] = int(row["rating"])

seen, unique = set(), []
for r in all_rows:
    key = r.get("upc") or r.get("link")
    if key and key in seen:
        continue
    seen.add(key)
    unique.append(r)

In [10]:
fields = ["ID", "title", "price", "rating", "availability", "link", "category"]

pd.DataFrame(unique, columns=fields).to_csv(CSV_FILE_NAME, index=False)

## Проерка CSV файла

In [11]:
df_books = pd.read_csv(CSV_FILE_NAME)

print(f'Количество строк: {len(df_books)}\n')
print(f'Доли пропусков:\n{df_books.isna().mean()}\n')
print(f'Первые 5 строк:\n{df_books.head(5).to_numpy()}')

Количество строк: 100

Доли пропусков:
ID              0.0
title           0.0
price           0.0
rating          0.0
availability    0.0
link            0.0
category        0.0
dtype: float64

Первые 5 строк:
[[1000 'A Light in the Attic' 51.77 3 'In stock'
  'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
  'Poetry']
 [999 'Tipping the Velvet' 53.74 1 'In stock'
  'https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html'
  'Historical Fiction']
 [998 'Soumission' 50.1 1 'In stock'
  'https://books.toscrape.com/catalogue/soumission_998/index.html'
  'Fiction']
 [997 'Sharp Objects' 47.82 4 'In stock'
  'https://books.toscrape.com/catalogue/sharp-objects_997/index.html'
  'Mystery']
 [996 'Sapiens: A Brief History of Humankind' 54.23 5 'In stock'
  'https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html'
  'History']]
